# Settlement-source anchor: retrospective outcome diagnostic

## tl;dr

Across 81 opportunity rows from three independent conditions, the frozen Chainlink anchor passed every freshness check but removed the only baseline-selected condition—a winner worth +0.314523 in decision-time one-share payoff—and selected no replacements. This is directionally negative, severely underpowered retrospective evidence. It neither rejects nor promotes the candidate and provides no profitability or A+ evidence.

## Context & Methods

The decision is whether the already-frozen `settlement_source_anchor_v1` is sufficiently plausible to justify two later 750-condition blocks—not whether it is profitable or A+. The source capture covers 24 BTC five-minute markets from 2026-07-15 06:50 through 08:50 UTC and predates the active forward block.

### Key assumptions and fixed estimands

- Only the registered fair spot and strike move from the Binance proxy to causal Chainlink current/open. The fee-aware edge floor remains `0.07`, the gross-edge stale cap remains `0.25`, current-source age remains at most 10 seconds, and open-source age remains at most 2 seconds. No neighbor thresholds are evaluated.
- The engine-generated opportunity file contains the first post–non-edge-gate opportunity per condition-second. Repeated seconds share a terminal label, so the decision statistic is the earliest qualifying opportunity per condition for each anchor. Event-weighted rows are descriptive only.
- The original Binance tape lacked one hour of causal preroll and had two gaps over 5 seconds. Official Binance one-second klines supply only that preroll and those 44 missing seconds; captured RTDS rows take precedence elsewhere.
- The captured Chainlink tape has an 8-second internal gap, so the full engine replay cannot use it as an exact settlement tape. The opportunity extraction uses the repaired Binance proxy for settlement; the retained resolution manifest independently shows proxy and official terminal directions agree on all 24 markets. Candidate fair values still use only the captured Chainlink tape and fail closed on stale rows.
- One terminal label was exposed while inspecting the resolution schema before these estimands were written. No aggregate or comparative result was inspected. This remains a retrospective diagnostic, never a preregistered test.
- Decision-time one-share payoff is `1 - ask - fee` when correct and `-ask - fee` otherwise. It ignores 202 ms book movement, L2 traversal, FOK failure, sizing, state feedback, and portfolio risk; it is not executable PnL.

## Data

### 1. Load and verify immutable inputs

In [1]:
import bisect
import csv
import gzip
import hashlib
import io
import json
import math
import os
import statistics
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path('/Users/ttoomm/Documents/PolyMomentum')
SNAPSHOTS = ROOT / 'deploy/promotions/evidence/strategy_registry/source_snapshots'
REGISTRY = ROOT / 'deploy/promotions/evidence/strategy_registry'
PATHS = {
    'opportunities': SNAPSHOTS / '20260721_settlement_anchor_historical_signal_opportunities.json.gz',
    'replay_report': SNAPSHOTS / '20260721_settlement_anchor_historical_signal_replay_report.json.gz',
    'chainlink': SNAPSHOTS / '20260721_settlement_anchor_historical_chainlink_btcusd.csv.gz',
    'resolution': SNAPSHOTS / '20260721_settlement_anchor_historical_resolution_manifest.json.gz',
    'captured_binance': SNAPSHOTS / '20260721_settlement_anchor_historical_binance_btcusdt_rtds.csv.gz',
    'binance_preroll': SNAPSHOTS / '20260721_binance_btcusdt_1s_preroll.csv.gz',
    'binance_gapfill': SNAPSHOTS / '20260721_binance_btcusdt_1s_gapfill.csv.gz',
    'capture_variant': REGISTRY / '20260715_binary_complement_capture_variant.json',
    'baseline_variant': REGISTRY / '20260721_settlement_source_anchor_baseline_variant.json',
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def load_gzip_json(path):
    with gzip.open(path, 'rt') as handle:
        return json.load(handle)

def sha256_gzip_payload(path):
    digest = hashlib.sha256()
    with gzip.open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def load_gzip_csv(path):
    with gzip.open(path, 'rt') as handle:
        return list(csv.DictReader(handle))

opportunity_doc = load_gzip_json(PATHS['opportunities'])
replay_report = load_gzip_json(PATHS['replay_report'])
resolution = load_gzip_json(PATHS['resolution'])
capture_variant = json.loads(PATHS['capture_variant'].read_text())
baseline_variant = json.loads(PATHS['baseline_variant'].read_text())

normalized_capture = dict(capture_variant)
normalized_capture['name'] = baseline_variant['name']
normalized_capture['min_edge'] = baseline_variant['min_edge']
normalized_capture.pop('degraded_after_drawdown_pct', None)
assert normalized_capture == baseline_variant
assert opportunity_doc['sampling'] == 'first_pre_edge_candidate_per_condition_utc_second'
assert opportunity_doc['continuous'] is True
assert opportunity_doc['latency_ms'] == 202
assert opportunity_doc['variant_count'] == 1
assert replay_report['variants'][0]['trades'] == 0
assert replay_report['variants'][0]['strategy']['params_hash'] == '34aa177f7ae8614814208cdd81ed74e09199007b924ee16b6e18dfa62fd49aa9'
assert resolution['stats']['terminal'] == 24
assert resolution['stats']['oracle_disagreements'] == 0
assert resolution['stats']['proxy_oracle_disagreements'] == 0
assert resolution['official_settlement_sources']['matched_to_btc_tape'] == 24
assert resolution['a_plus_gate']['settlement_alignment_ready'] is False
preroll_rows = load_gzip_csv(PATHS['binance_preroll'])
gapfill_rows = load_gzip_csv(PATHS['binance_gapfill'])
assert len(preroll_rows) == 3572
assert int(preroll_rows[0]['timestamp_ms']) == 1784094600000
assert int(preroll_rows[-1]['timestamp_ms']) == 1784098171000
assert len(gapfill_rows) == 44
assert {row['source'] for row in preroll_rows} == {'binance_btcusdt_klines_1s_preroll'}
assert {row['source'] for row in gapfill_rows} == {'binance_btcusdt_klines_1s_gapfill'}

source_hashes = {name: sha256_file(path) for name, path in PATHS.items()}
source_uncompressed_hashes = {
    name: sha256_gzip_payload(PATHS[name])
    for name in ('opportunities', 'replay_report', 'chainlink', 'resolution', 'captured_binance', 'binance_preroll', 'binance_gapfill')
}
assert source_uncompressed_hashes['opportunities'] == 'e23bdbb7da926b67840bc917b7db688ffa5c1e91aedf8ee302d92559da38c0ee'
assert source_uncompressed_hashes['replay_report'] == '4a842019976238d3c68bc357e4f73331ff64ac91b4783bcd70ceacbb294c7915'
assert source_uncompressed_hashes['chainlink'] == 'caa86f114881e6ca79a83ce1be1532623af46ad9d869ebeef95c16175a93aa5d'
assert source_uncompressed_hashes['resolution'] == 'b94b8a096d045c25ffe42b3f419e689c38aab9a193b8309c77e3eef551589322'
assert source_uncompressed_hashes['captured_binance'] == 'a35f0831be0ae2880435ac4e1051f6e40164e2f38d46bcc0b35457797800b9de'
print({
    'opportunity_rows': opportunity_doc['row_count'],
    'opportunity_conditions': opportunity_doc['condition_count'],
    'terminal_markets': resolution['stats']['terminal'],
    'official_vs_proxy_terminal_disagreements': resolution['stats']['proxy_oracle_disagreements'],
    'source_files_verified': len(source_hashes),
})

{'opportunity_rows': 81, 'opportunity_conditions': 3, 'terminal_markets': 24, 'official_vs_proxy_terminal_disagreements': 0, 'source_files_verified': 9}


## Results

### 2. Recompute baseline fair value and substitute only the official anchor

In [2]:
def load_tick_tape(path):
    first_by_timestamp = {}
    with gzip.open(path, 'rt') as handle:
        for row in csv.DictReader(handle):
            timestamp_ms = int(float(row['timestamp_ms']))
            price = float(row['price'])
            if price > 0 and math.isfinite(price):
                first_by_timestamp.setdefault(timestamp_ms, price)
    pairs = sorted(first_by_timestamp.items())
    return [timestamp for timestamp, _ in pairs], [price for _, price in pairs]

def at(tape, timestamp_ms):
    timestamps, prices = tape
    index = bisect.bisect_right(timestamps, timestamp_ms) - 1
    if index < 0:
        return None, None
    return prices[index], timestamp_ms - timestamps[index]

def erf_abramowitz_stegun(value):
    coefficients = (0.254829592, -0.284496736, 1.421413741, -1.453152027, 1.061405429)
    sign = 1.0 if value >= 0 else -1.0
    value = abs(value)
    t = 1.0 / (1.0 + 0.3275911 * value)
    polynomial = (((((coefficients[4] * t + coefficients[3]) * t + coefficients[2]) * t + coefficients[1]) * t + coefficients[0]) * t)
    return sign * (1.0 - polynomial * math.exp(-value * value))

def binary_option_up_probability(spot, strike, minutes_remaining, volatility):
    days_remaining = minutes_remaining / 1440.0
    years = days_remaining / 365.25
    d2 = (math.log(spot / strike) + (0.05 - 0.5 * volatility * volatility) * years) / (volatility * math.sqrt(years))
    probability = 0.5 * (1.0 + erf_abramowitz_stegun(d2 / math.sqrt(2.0)))
    return min(max(probability, 0.01), 0.99)

chainlink = load_tick_tape(PATHS['chainlink'])
windows = {market['condition_id']: market for market in resolution['markets']}
records = []
baseline_recompute_errors = []
for wrapped in opportunity_doc['rows']:
    opportunity = wrapped['opportunity']
    decision = opportunity['decision']
    condition_id = opportunity['condition_id']
    market = windows[condition_id]
    proxy_up = binary_option_up_probability(
        opportunity['btc_price'],
        opportunity['open_btc'],
        decision['minutes_remaining'],
        opportunity['decision_volatility'],
    )
    proxy_fair = proxy_up if decision['direction'] == 'up' else 1.0 - proxy_up
    baseline_recompute_errors.append(abs(proxy_fair - decision['fair_value']))
    decision_timestamp_ms = int(opportunity['decision_timestamp_s'] * 1000.0)
    official_current, current_age_ms = at(chainlink, decision_timestamp_ms)
    official_open, open_age_ms = at(chainlink, market['open_ts_s'] * 1000)
    source_fresh = (
        official_current is not None
        and official_open is not None
        and current_age_ms <= 10_000
        and open_age_ms <= 2_000
    )
    if source_fresh:
        official_up = binary_option_up_probability(
            official_current,
            official_open,
            decision['minutes_remaining'],
            opportunity['decision_volatility'],
        )
        official_fair = official_up if decision['direction'] == 'up' else 1.0 - official_up
        official_gross_edge = official_fair - decision['market_price']
        official_edge = official_gross_edge - decision['entry_fee_per_share']
    else:
        official_fair = official_gross_edge = official_edge = None
    baseline_eligible = decision['edge'] >= 0.07 and decision['gross_edge'] <= 0.25
    official_eligible = bool(source_fresh and official_edge >= 0.07 and official_gross_edge <= 0.25)
    one_share_pnl = (1.0 if opportunity['won'] else 0.0) - decision['market_price'] - decision['entry_fee_per_share']
    records.append({
        'condition_id': condition_id,
        'decision_timestamp_s': opportunity['decision_timestamp_s'],
        'direction': decision['direction'],
        'won': opportunity['won'],
        'ask': decision['market_price'],
        'fee_per_share': decision['entry_fee_per_share'],
        'one_share_pnl': one_share_pnl,
        'baseline_fair': decision['fair_value'],
        'official_fair': official_fair,
        'baseline_edge': decision['edge'],
        'official_edge': official_edge,
        'edge_delta': None if official_edge is None else official_edge - decision['edge'],
        'current_age_ms': current_age_ms,
        'open_age_ms': open_age_ms,
        'source_fresh': source_fresh,
        'baseline_eligible': baseline_eligible,
        'official_eligible': official_eligible,
    })

assert max(baseline_recompute_errors) < 1e-12
assert len(records) == opportunity_doc['row_count']
assert len({record['condition_id'] for record in records}) == opportunity_doc['condition_count']
print({'maximum_baseline_fair_recompute_error': max(baseline_recompute_errors), 'records': len(records)})

{'maximum_baseline_fair_recompute_error': 0.0, 'records': 81}


### 3. Compare the earliest qualifying decision per condition

In [3]:
def earliest_qualifying(records, eligibility_field):
    selected = {}
    for record in sorted(records, key=lambda row: (row['decision_timestamp_s'], row['condition_id'])):
        if record[eligibility_field] and record['condition_id'] not in selected:
            selected[record['condition_id']] = record
    return selected

def selection_summary(selected):
    values = list(selected.values())
    wins = sum(record['won'] for record in values)
    pnl = sum(record['one_share_pnl'] for record in values)
    return {
        'selected_conditions': len(values),
        'wins': wins,
        'losses': len(values) - wins,
        'win_rate': wins / len(values) if values else None,
        'decision_time_one_share_pnl': pnl,
        'average_one_share_pnl': pnl / len(values) if values else None,
    }

baseline_selected = earliest_qualifying(records, 'baseline_eligible')
official_selected = earliest_qualifying(records, 'official_eligible')
all_condition_ids = sorted(windows)
condition_comparison = []
for condition_id in all_condition_ids:
    baseline = baseline_selected.get(condition_id)
    official = official_selected.get(condition_id)
    baseline_pnl = baseline['one_share_pnl'] if baseline else 0.0
    official_pnl = official['one_share_pnl'] if official else 0.0
    same_decision = (
        baseline is None and official is None
    ) or (
        baseline is not None
        and official is not None
        and baseline['direction'] == official['direction']
        and baseline['decision_timestamp_s'] == official['decision_timestamp_s']
    )
    condition_comparison.append({
        'condition_id': condition_id,
        'baseline_selected': baseline is not None,
        'official_selected': official is not None,
        'same_decision': same_decision,
        'baseline_won': baseline['won'] if baseline else None,
        'official_won': official['won'] if official else None,
        'baseline_one_share_pnl': baseline_pnl,
        'official_one_share_pnl': official_pnl,
        'official_minus_baseline_one_share_pnl': official_pnl - baseline_pnl,
    })

fresh_records = [record for record in records if record['source_fresh']]
event_results = {
    'rows': len(records),
    'conditions': len({record['condition_id'] for record in records}),
    'official_source_fresh_rows': len(fresh_records),
    'official_source_coverage': len(fresh_records) / len(records),
    'baseline_edge_pass_rows': sum(record['baseline_eligible'] for record in records),
    'official_edge_pass_rows': sum(record['official_eligible'] for record in records),
    'eligibility_disagreement_rows': sum(record['baseline_eligible'] != record['official_eligible'] for record in records),
    'edge_delta': {
        'minimum': min(record['edge_delta'] for record in fresh_records),
        'median': statistics.median(record['edge_delta'] for record in fresh_records),
        'maximum': max(record['edge_delta'] for record in fresh_records),
        'mean': statistics.fmean(record['edge_delta'] for record in fresh_records),
    },
}
condition_results = {
    'baseline': selection_summary(baseline_selected),
    'official': selection_summary(official_selected),
    'changed_decisions': sum(not row['same_decision'] for row in condition_comparison),
    'baseline_only_conditions': sum(row['baseline_selected'] and not row['official_selected'] for row in condition_comparison),
    'official_only_conditions': sum(row['official_selected'] and not row['baseline_selected'] for row in condition_comparison),
    'paired_one_share_pnl_delta_across_all_24_conditions': sum(row['official_minus_baseline_one_share_pnl'] for row in condition_comparison),
}
summary = {'event_level_descriptive': event_results, 'condition_level_primary': condition_results}
print(json.dumps(summary, indent=2))
print('Changed condition decisions:')
for row in condition_comparison:
    if not row['same_decision']:
        print(row)

{
  "event_level_descriptive": {
    "rows": 81,
    "conditions": 3,
    "official_source_fresh_rows": 81,
    "official_source_coverage": 1.0,
    "baseline_edge_pass_rows": 1,
    "official_edge_pass_rows": 0,
    "eligibility_disagreement_rows": 1,
    "edge_delta": {
      "minimum": -0.07061215526725984,
      "median": -0.0035103351267746774,
      "maximum": 0.024897479897172947,
      "mean": -0.004120676499047977
    }
  },
  "condition_level_primary": {
    "baseline": {
      "selected_conditions": 1,
      "wins": 1,
      "losses": 0,
      "win_rate": 1.0,
      "decision_time_one_share_pnl": 0.31452299999999994,
      "average_one_share_pnl": 0.31452299999999994
    },
    "official": {
      "selected_conditions": 0,
      "wins": 0,
      "losses": 0,
      "win_rate": null,
      "decision_time_one_share_pnl": 0,
      "average_one_share_pnl": null
    },
    "changed_decisions": 1,
    "baseline_only_conditions": 1,
    "official_only_conditions": 0,
    "paired_one

### 4. Inspect bounded edge displacement without implying independent observations

In [4]:
largest_displacements = sorted(
    fresh_records,
    key=lambda record: abs(record['edge_delta']),
    reverse=True,
)[:12]
for record in largest_displacements:
    print({
        'condition_id': record['condition_id'],
        'timestamp_s': record['decision_timestamp_s'],
        'direction': record['direction'],
        'won': record['won'],
        'baseline_edge': round(record['baseline_edge'], 6),
        'official_edge': round(record['official_edge'], 6),
        'edge_delta': round(record['edge_delta'], 6),
    })

{'condition_id': '0x9759a66a41fc6625e47fc7fe5832321b9f47e99416761114ea812993bae111d7', 'timestamp_s': 1784099546.0, 'direction': 'up', 'won': True, 'baseline_edge': 0.072011, 'official_edge': 0.001399, 'edge_delta': -0.070612}
{'condition_id': '0x9759a66a41fc6625e47fc7fe5832321b9f47e99416761114ea812993bae111d7', 'timestamp_s': 1784099547.009, 'direction': 'up', 'won': True, 'baseline_edge': -0.024221, 'official_edge': -0.085368, 'edge_delta': -0.061147}
{'condition_id': '0x9759a66a41fc6625e47fc7fe5832321b9f47e99416761114ea812993bae111d7', 'timestamp_s': 1784099563.0, 'direction': 'up', 'won': True, 'baseline_edge': 0.006146, 'official_edge': -0.032648, 'edge_delta': -0.038794}
{'condition_id': '0x919207d79db85c5fa635f31113bebf1969e8a50be9dd639a90bd6746fa934b61', 'timestamp_s': 1784098959.6, 'direction': 'down', 'won': True, 'baseline_edge': -0.049063, 'official_edge': -0.024165, 'edge_delta': 0.024897}
{'condition_id': '0x919207d79db85c5fa635f31113bebf1969e8a50be9dd639a90bd6746fa934b61

## Takeaways

The official anchor changed one of three condition-level decisions: baseline edge `0.072011` selected a winning UP share, whereas official-anchor edge `0.001399` correctly failed the frozen `0.07` floor. That reduced the diagnostic one-share payoff by `0.314523` across all 24 captured conditions. The result is reproducible from hash-pinned inputs, but only three conditions reached the strategy gate and the repaired proxy tape was required for exact replay labels. Keep the forward contract frozen; do not promote, retune, or claim profitability. Two disjoint 750-condition blocks remain mandatory if later evidence still justifies pursuing this mechanism.

In [5]:
EVIDENCE_PATH = REGISTRY / '20260721_settlement_source_anchor_historical_outcome_diagnostic.json'
pnl_delta = condition_results['paired_one_share_pnl_delta_across_all_24_conditions']
changed_decisions = condition_results['changed_decisions']
if changed_decisions == 0:
    status = 'RETROSPECTIVE_SIGNAL_GATED_DIAGNOSTIC_NO_CONDITION_DECISION_CHANGE'
    directional_assessment = 'NO_OBSERVED_SIGNAL_GATED_SELECTIVITY'
elif pnl_delta > 0:
    status = 'RETROSPECTIVE_SIGNAL_GATED_DIAGNOSTIC_DIRECTIONALLY_POSITIVE_INADEQUATE_SUPPORT'
    directional_assessment = 'DIRECTIONALLY_POSITIVE_BUT_UNDERPOWERED'
elif pnl_delta < 0:
    status = 'RETROSPECTIVE_SIGNAL_GATED_DIAGNOSTIC_DIRECTIONALLY_NEGATIVE_INADEQUATE_SUPPORT'
    directional_assessment = 'DIRECTIONALLY_NEGATIVE_BUT_UNDERPOWERED'
else:
    status = 'RETROSPECTIVE_SIGNAL_GATED_DIAGNOSTIC_ECONOMICALLY_NEUTRAL_INADEQUATE_SUPPORT'
    directional_assessment = 'ECONOMICALLY_NEUTRAL_AND_UNDERPOWERED'

evidence = {
    'schema_version': 1,
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'mechanism_id': 'settlement_source_anchor_v1',
    'status': status,
    'decision_question': 'Does official Chainlink anchoring improve the outcome alignment of actual post-non-edge-gate strategy opportunities in the historical July 15 capture?',
    'authority': {
        'diagnostic_only': True,
        'retrospective_labels_loaded': True,
        'one_label_observed_before_estimands_locked': True,
        'active_forward_block_loaded': False,
        'active_forward_strategy_metrics_loaded': False,
        'promotion_credit': False,
        'exact_replay_evidence': False,
    },
    'frozen_comparison': {
        'baseline': 'primary_v6_volfloor_300',
        'candidate': 'settlement_source_anchor_v1',
        'only_changed_inputs': ['fair-value current spot', 'fair-value five-minute strike'],
        'minimum_fee_aware_edge': 0.07,
        'gross_edge_stale_cap': 0.25,
        'official_current_max_age_ms': 10000,
        'official_open_max_age_ms': 2000,
        'alternate_thresholds_tested': 0,
    },
    'source_authority': {
        'window_utc': '2026-07-15T06:50:00Z through 2026-07-15T08:50:00Z',
        'captured_markets': 24,
        'engine_opportunity_rows': opportunity_doc['row_count'],
        'engine_opportunity_conditions': opportunity_doc['condition_count'],
        'official_vs_proxy_terminal_direction_disagreements': resolution['stats']['proxy_oracle_disagreements'],
        'binance_market_data_documentation': 'https://developers.binance.com/en/docs/products/spot/rest-api',
        'binance_public_market_data_endpoint': 'https://data-api.binance.vision/api/v3/klines',
        'source_sha256': source_hashes,
        'source_uncompressed_sha256': source_uncompressed_hashes,
    },
    'methodology': {
        'unit_of_decision': 'earliest edge-qualified post-non-edge-gate opportunity per condition and anchor',
        'event_rows_are_independent': False,
        'decision_time_one_share_pnl_is_executable': False,
        'baseline_fair_value_max_recompute_error': max(baseline_recompute_errors),
    },
    'results': summary,
    'changed_condition_rows': [row for row in condition_comparison if not row['same_decision']],
    'assessment': {
        'directional_assessment': directional_assessment,
        'independent_strategy_opportunity_conditions': opportunity_doc['condition_count'],
        'minimum_future_conditions_per_registered_block': 750,
        'candidate_action': 'KEEP_FROZEN_FORWARD_CONTRACT_UNCHANGED; DO_NOT PROMOTE, RETUNE, OR CLAIM PROFITABILITY',
        'reason': 'Only three historical conditions reached the non-edge strategy gate, the signal tape required external gap repair, and the official tape cannot satisfy exact settlement continuity. The diagnostic can reject an obviously harmful mechanism but cannot validate A+ economics.',
    },
    'limitations': [
        'Only three of 24 captured conditions produced post-non-edge-gate strategy opportunities.',
        'Repeated opportunity seconds share terminal labels and are not independent observations.',
        'The signal tape uses official Binance one-second klines for the missing preroll and 44 internal gap seconds; captured RTDS takes precedence elsewhere.',
        'The captured Chainlink tape fails the harness five-second settlement continuity check, so engine opportunity labels use the repaired Binance proxy; the retained resolution manifest shows identical official and proxy terminal directions for all 24 markets.',
        'Decision-time one-share payoff excludes latency, L2 traversal, fill failure, sizing, exposure state, breaker state, and tail clustering.',
        'The same 24-market source capture motivated the candidate, so this is retrospective diagnostic evidence with zero promotion credit.',
    ],
    'decision': {
        'candidate_rejected': False,
        'candidate_promoted': False,
        'future_disjoint_blocks_still_required': 2,
        'runtime_change_authorized': False,
        'paper_or_live_trading_authorized': False,
        'profitability_claim': False,
        'a_plus_claim': False,
    },
}
temporary = EVIDENCE_PATH.with_name(f'{EVIDENCE_PATH.name}.tmp.{os.getpid()}')
temporary.write_text(json.dumps(evidence, indent=2, sort_keys=True) + '\n')
temporary.replace(EVIDENCE_PATH)
print({'artifact': str(EVIDENCE_PATH.relative_to(ROOT)), 'status': status, 'pnl_delta': pnl_delta, 'changed_decisions': changed_decisions})

{'artifact': 'deploy/promotions/evidence/strategy_registry/20260721_settlement_source_anchor_historical_outcome_diagnostic.json', 'status': 'RETROSPECTIVE_SIGNAL_GATED_DIAGNOSTIC_DIRECTIONALLY_NEGATIVE_INADEQUATE_SUPPORT', 'pnl_delta': -0.31452299999999994, 'changed_decisions': 1}
